[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/interactive_control.ipynb)

# Interactive Control Playground

Use sliders to tune PD gains in real-time and see how they affect pendulum behavior.

**Requirements:** `ipywidgets` (included in Colab and JupyterLab)

In [1]:
!pip install -q mujoco mediapy ipywidgets matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.1 MB/s eta 0:00:00


In [2]:
import os
os.environ['MUJOCO_GL'] = 'egl'
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
XML = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2" dir="0 0 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".9 .9 .9 1"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" mass="1" rgba="0.2 0.6 0.9 1"/>
    </body>
  </worldbody>
  <actuator>
    <motor joint="hinge" ctrlrange="-10 10"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(XML)
print(f"Model loaded: {model.nq} DOF, {model.nu} actuators")

Model loaded: 1 DOF, 1 actuators


## Simulate with Given Gains

This function runs a simulation and returns the time-series data:

In [4]:
def simulate_pd(kp, kd, target_deg=60.0, duration=5.0):
    """Run PD simulation and return time, angle, torque arrays."""
    data = mujoco.MjData(model)
    target = np.radians(target_deg)

    times, angles, torques = [], [], []
    while data.time < duration:
        error = target - data.qpos[0]
        torque = kp * error - kd * data.qvel[0]
        data.ctrl[0] = np.clip(torque, -10, 10)

        times.append(data.time)
        angles.append(np.degrees(data.qpos[0]))
        torques.append(data.ctrl[0])
        mujoco.mj_step(model, data)

    return np.array(times), np.array(angles), np.array(torques)

## Interactive Slider Widget

Drag the sliders to change `kp`, `kd`, and target angle. The plot updates instantly!

In [5]:
output = widgets.Output()

kp_slider = widgets.FloatSlider(value=10, min=0, max=50, step=0.5,
                                 description='Kp (stiffness):', style={'description_width': '120px'},
                                 layout=widgets.Layout(width='500px'))
kd_slider = widgets.FloatSlider(value=2, min=0, max=20, step=0.2,
                                 description='Kd (damping):', style={'description_width': '120px'},
                                 layout=widgets.Layout(width='500px'))
target_slider = widgets.FloatSlider(value=60, min=-90, max=90, step=5,
                                     description='Target (deg):', style={'description_width': '120px'},
                                     layout=widgets.Layout(width='500px'))

def update_plot(change=None):
    kp = kp_slider.value
    kd = kd_slider.value
    target = target_slider.value

    times, angles, torques = simulate_pd(kp, kd, target)

    with output:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

        ax1.plot(times, angles, 'b-', linewidth=1.5)
        ax1.axhline(target, color='r', linestyle='--', alpha=0.7, label=f'Target = {target}°')
        ax1.set_ylabel('Angle (deg)')
        ax1.set_ylim(-100, 100)
        ax1.legend(loc='upper right')
        ax1.grid(True, alpha=0.3)
        ax1.set_title(f'PD Control — Kp={kp:.1f}, Kd={kd:.1f}')

        # Classify response
        overshoot = max(angles) - target if max(angles) > target else 0
        ss_error = abs(angles[-1] - target)
        ax1.text(0.02, 0.95, f'Overshoot: {overshoot:.1f}°\nSS Error: {ss_error:.2f}°',
                 transform=ax1.transAxes, va='top', fontsize=9,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        ax2.plot(times, torques, 'orange', linewidth=1.5)
        ax2.axhline(0, color='k', linewidth=0.5)
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel('Torque (Nm)')
        ax2.set_ylim(-11, 11)
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# Attach observers
kp_slider.observe(update_plot, names='value')
kd_slider.observe(update_plot, names='value')
target_slider.observe(update_plot, names='value')

# Display
display(widgets.VBox([kp_slider, kd_slider, target_slider, output]))
update_plot()  # Initial plot

## Render a Video

Once you've found gains you like, render a video with those settings:

In [9]:
def render_pd_video(kp, kd, target_deg=60.0, duration=4.0, fps=30):
    """Render a video of the pendulum with given PD gains."""
    data = mujoco.MjData(model)
    target = np.radians(target_deg)
    renderer = mujoco.Renderer(model, height=300, width=400)

    frames = []
    steps_per_frame = int(1.0 / (model.opt.timestep * fps))

    while data.time < duration:
        data.ctrl[0] = np.clip(kp * (target - data.qpos[0]) - kd * data.qvel[0], -10, 10)
        mujoco.mj_step(model, data)

        if int(data.time / model.opt.timestep) % steps_per_frame == 0:
            renderer.update_scene(data)
            frames.append(renderer.render().copy())

    renderer.close()
    return frames

# Use current slider values
frames = render_pd_video(kp_slider.value, kd_slider.value, target_slider.value)
media.show_video(frames, fps=30)

## Exercises

1. **Find critically damped gains** — minimize overshoot while reaching target quickly
2. **What happens with Kd=0?** — pure proportional control
3. **What happens with very high Kp?** — notice the torque saturates at ±10 Nm
4. **Try negative Kd** — what goes wrong? (hint: positive feedback)

**Key insight:** The ratio $\zeta = \frac{K_d}{2\sqrt{K_p}}$ determines damping:
- $\zeta < 1$: underdamped (oscillates)
- $\zeta = 1$: critically damped (fastest without overshoot)
- $\zeta > 1$: overdamped (slow approach)